# Day 8: Autonomous ReAct Agent & Recursive Language Model (RLM)
**Config Source:** `.env` + `configs/app_config.yaml` (`engine/settings.py`) — Zero hardcoded URLs in notebook cells.


### Cell 1 — Centralized Settings & Modular Engine Bootstrap


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

from engine.settings import SETTINGS
from engine.router import ModelRouter
from engine.qdrant_store import QdrantVectorStore
from engine.use_case_loader import UseCaseLoader
from engine.visualizer import StoryVisualizer
from engine.debugger import TraceDebugger

# Import PromptRegistry safely regardless of module filename (prompts.py or prompt_registry.py)
try:
    from engine.prompts import PromptRegistry
except ImportError:
    from engine.prompt_registry import PromptRegistry

# All URLs, paths, domain names, and cloud toggles come from SETTINGS (.env + configs/app_config.yaml)
ACTIVE_DOMAIN = SETTINGS.active_domain  # Override here or via BOOTCAMP_DOMAIN in .env

loader = UseCaseLoader(domain=ACTIVE_DOMAIN)
llm = ModelRouter(local_model_path=SETTINGS.local_model_path, enable_vertex=SETTINGS.enable_vertex_ai)
qdrant = QdrantVectorStore(collection_name=loader.collection_name, path=SETTINGS.qdrant_db_path)
prompts = PromptRegistry()
viz = StoryVisualizer()
dbg = TraceDebugger(debug_mode=False, enable_phoenix=False)

RUNBOOKS = loader.load_runbooks()
INCIDENTS = loader.load_incidents()
qdrant.upsert_documents(RUNBOOKS)

print(f"[Ready] Domain: {loader.domain} | LLM: {llm.provider} | VectorDB: {qdrant.mode} ({loader.collection_name})")


### Cell 2 — Production Incident Selector (0..9)


In [ ]:
# Select any of the 10 production incidents in the active domain (0..9)
INCIDENT_INDEX = 0
ACTIVE_INCIDENT = INCIDENTS[INCIDENT_INDEX]

print(f"Selected Incident: [{ACTIVE_INCIDENT['incident_id']}] {ACTIVE_INCIDENT['title']} (Severity: {ACTIVE_INCIDENT['severity']})")
print("Raw Log:", ACTIVE_INCIDENT["raw_log"])


### Cell 3 — Engineering Concept Brief
**Concept: Tool-Using ReAct Agent + RLM Long-Log Chunking**
Correlates live metrics + Qdrant runbooks and demonstrates `rlm_recursive_log_analyzer` without hardcoded endpoints.


### Cell 4 — Traced Execution & X-Ray Visual Storyboard


In [ ]:
from engine.a2a_skills import rlm_recursive_log_analyzer

dbg.start_run("day08_agent")
with dbg.start_span(f"Day8_ReActAgent_{ACTIVE_INCIDENT['incident_id']}", span_kind="AGENT") as root_span:
    t0 = time.perf_counter()
    metrics_obs = ACTIVE_INCIDENT["metrics"]
    t_m = round((time.perf_counter() - t0) * 1000, 2)
    dbg.record_step("tool_fetch_metrics", ACTIVE_INCIDENT["service"], metrics_obs, t_m, span_kind="TOOL")

    t1 = time.perf_counter()
    rb_hit = qdrant.search(ACTIVE_INCIDENT["raw_log"], top_k=1)[0]
    t_q = round((time.perf_counter() - t1) * 1000, 2)
    dbg.record_step("tool_search_qdrant", ACTIVE_INCIDENT["raw_log"], rb_hit, t_q, span_kind="RETRIEVER")

    p_cfg = prompts.load("day08_react_agent")
    agent_prompt = prompts.render(
        p_cfg["user_template"],
        incident_text=ACTIVE_INCIDENT["raw_log"],
        telemetry_obs=json.dumps(metrics_obs),
        runbook_obs=rb_hit["text"],
    )
    decision = llm.generate(prompt=agent_prompt, system_instruction=p_cfg["system_prompt"], temperature=0.1, max_tokens=160)
    dbg.record_step("agent_commander_plan", agent_prompt, decision["output"], decision["latency_ms"], span_kind="LLM")

viz.render_story(
    title=f"Day 8 Autonomous ReAct Agent — {ACTIVE_INCIDENT['incident_id']}",
    stages=[
        {"name": "Tool 1: Live Telemetry Metrics", "badge": "TOOL", "latency_ms": t_m, "summary": ACTIVE_INCIDENT["service"], "payload": metrics_obs},
        {"name": "Tool 2: Qdrant Runbook Lookup", "badge": "RETRIEVER", "latency_ms": t_q, "summary": rb_hit["metadata"]["runbook_id"], "payload": rb_hit["text"]},
        {"name": "Agent Remediation Plan", "badge": "AGENT PLAN", "latency_ms": decision["latency_ms"], "summary": "Synthesized plan", "payload": decision["output"]}
    ],
    phoenix_url=dbg.phoenix_url
)


### Cell 5 — Guarded Cloud Parity Check (`SETTINGS.enable_vertex_ai`)


In [ ]:
if SETTINGS.enable_vertex_ai:
    print("Vertex AI Agent Engine deployment active.")
else:
    print("ENABLE_VERTEX_AI=false -> Local ReAct Agent verified.")


### Cell 6 — Automated Validation Gate


In [ ]:
assert len(dbg.runs["day08_agent"]) == 3
print("[PASS] Day 8 Agentic Workflows Gate passed!")
